# **Data Analysis for AI BSc Thesis Educational Agents** 
by Sandy Wong


In [ ]:
# Import packages
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import shapiro

In [ ]:
# Plot settings
sns.set_palette("colorblind")
sns.set_context("paper", font_scale=1.5)
plt.rc(group="font", family="serif")
WIDTH = 11.69 # size of A4

## Load and clean data
**Note**: The original data files are not included in the public repository due to privacy and ethical considerations.
To run this notebook, place the data files in the local data/ folder.

In [ ]:
# Load data
df_chatbot = pd.read_csv('data/chatbot_data.csv')
df_vh = pd.read_csv('data/vh_data.csv')

In [ ]:
# Remove irrelevant columns
df_chatbot.drop(columns=["id", "seed", "submitdate", "lastpage", "startlanguage", "welcome", "firstTime", "instruction", "chatbotDone", "interaction"], inplace=True)
df_vh.drop(columns=["id", "seed", "submitdate", "lastpage", "startlanguage", "welcome", "firstTime", "instruction", "vhDone", "interaction"], inplace=True)

In [ ]:
# Merge dataframes
df = pd.merge(df_chatbot, df_vh, on="ID", how="inner", suffixes=("_chatbot", "_vh"))

In [ ]:
# Create cleaned dataframe
df_clean = pd.DataFrame({"ID": df["ID"]})

# Define shared columns
shared_cols = ["gender", "gender[other]", "age", "education", "education[other]", "priorJAP"]

# Keep only one column for each shared column
for col in shared_cols:
    df_clean[col] = df[f"{col}_chatbot"].combine_first(df[f"{col}_vh"])

df_clean.loc[df_clean["gender"].eq("-oth-"), "gender"] = df_clean["gender[other]"]
df_clean.loc[df_clean["education"].eq("-oth-"), "education"] = df_clean["education[other]"]

df_clean.drop(columns=["gender[other]", "education[other]"], inplace=True)

# Add prior experience columns
df_clean["prior_chatbot"] = df["priorC"]
df_clean["prior_vh"] = df["priorVH"]

## Demographics Summary

In [ ]:
# Gender counts
print(df_clean["gender"].value_counts())
print("\n")

# Age
age_mean = df_clean['age'].mean()
age_std = df_clean['age'].std()
age_range = [df_clean['age'].min(), df_clean['age'].max()]
print("age")
print(f"M = {age_mean:.2f}, SD = {age_std:.2f}, range = {age_range[0]}–{age_range[1]}")
print("\n")

# Education
print(df_clean["education"].value_counts())
print("\n")

# Prior experience
print(df_clean["prior_chatbot"].value_counts())
print("\n")
print(df_clean["prior_vh"].value_counts())
print("\n")


# Prior knowledge of Japan's political system
print(df_clean["priorJAP"].value_counts())

## Create scores

In [ ]:
construct_items = {
    "SP": ["SP1", "SP2", "SP3"],
    "UEP": ["UEP1", "UEP2", "UEP3", "UEP4"],
    "UE": ["UE1", "UE2", "UE3"],
    "AU": ["AU1", "AU2", "AU3"],
    "MC": ["MC1", "MC2", "MC3", "MC4"],
    "UAA": ["UAA1", "UAA2", "UAA3"],
}

prefixes = {
    "chatbot": "c",
    "virtual_human": "vh"
}

construct_abb = {
    "SP": "Social Presence",
    "UEP": "User's Emotion Presence",
    "UE": "User's Engagement",
    "AU": "Agent's Usability",
    "MC": "Metacognition",
    "UAA": "User Acceptance of the Agent",
}

hypothesis_constructs = ["SP", "UEP", "UE", "AU"]

# Define column name for eacch construct for each condition
construct_cols = [
    f"{construct}_{condition}"
    for construct in construct_items
    for condition in ["chatbot", "virtual_human"]
]

# Average constructs and put them in dataframe
for condition, prefix in prefixes.items():
    for construct, items in construct_items.items():
        cols = [f"{prefix}{construct}[{prefix}{item}]" for item in items]
        
        # Convert to -3–+3 
        if construct == "MC":
            correct_construct = df[cols] - 4  # 1–7 scale (mistake from questionnaire)
        else:
            correct_construct = df[cols] - 3  # 0–6 scale

        # Reverse-score UAA3 only
        if construct == "UAA":
            correct_construct[f"{prefix}{construct}[{prefix}UAA3]"] = -correct_construct[f"{prefix}{construct}[{prefix}UAA3]"]

        # Average construct (items)
        df_clean[f"{construct}_{condition}"] = correct_construct.mean(axis=1)

# Create dataframe with only paired data of constructs and drop NA
df_constructs = df_clean[construct_cols].dropna()

## Descriptive Statistics

In [ ]:
df_constructs.describe()

In [ ]:
# Create boxplots of all constructs by condition

fig, axes = plt.subplots(2, 3, figsize=(WIDTH, 8))
axes = axes.flatten()

for i, construct in enumerate(construct_items.keys()):
    paired_data = df_constructs[[f"{construct}_virtual_human", f"{construct}_chatbot"]].rename(columns={
        f"{construct}_virtual_human": "Virtual Human",
        f"{construct}_chatbot": "Chatbot"
    })

    sns.boxplot(data=paired_data, ax=axes[i])
    
    axes[i].set_title(f"{construct_abb[construct]}")
    axes[i].set_ylim(-3.5, 3.5)

fig.supylabel("Score")
fig.supxlabel("Conversational Agent")
plt.tight_layout()
plt.savefig("figures/boxplot_all.pdf", bbox_inches="tight", transparent=True, pad_inches=0.5)
plt.show()

In [ ]:
# Create boxplots of main constructs by condition
fig, axes = plt.subplots(2, 2, figsize=(WIDTH, WIDTH))
axes = axes.flatten()

for i, construct in enumerate(hypothesis_constructs):
    paired_data = df_constructs[[f"{construct}_virtual_human", f"{construct}_chatbot"]].rename(columns={
        f"{construct}_virtual_human": "Virtual Human",
        f"{construct}_chatbot": "Chatbot"
    })

    sns.boxplot(data=paired_data, ax=axes[i])
    
    axes[i].set_title(f"{construct_abb[construct]}")
    axes[i].set_ylim(-3.5, 3.5)

fig.supylabel("Score")
fig.supxlabel("Conversational Agent")
plt.tight_layout()
plt.savefig("figures/boxplot_main.pdf", bbox_inches="tight", transparent=True, pad_inches=0.5)
plt.show()

## Paired T-test Assumption Check
The paired sample t-test has four main assumptions:
- The dependent variable must be continuous (interval/ratio): yes, each construct is an average of items rated on a Likert scale from -3 to 3.
- The observations are independent of one another: yes, participants do not influence each other
- The dependent variable should approximately distribute normally: see below
- The dependent variable should not contain any outliers: above in boxplots you can see there are few for UE and AU. I will note this as a limitation, since it is a small dataset and only one or two outliers.
  
The observations are defined as the differences between two sets of values. Hence, each assumption refers to these differences, and not the original data values. 

### Outlier Detection

In [ ]:
# Visualize outliers and detect outliers by IQR method
fig, axes = plt.subplots(2, 2, figsize=(WIDTH, 8))
axes = axes.flatten()
df_outliers_removed = df_constructs.copy()

for i, construct in enumerate(hypothesis_constructs):
    diff = df_constructs[f"{construct}_virtual_human"] - df_constructs[f"{construct}_chatbot"] # Difference of construct score between conditions
    
    Q1, Q3 = diff.quantile(0.25), diff.quantile(0.75)
    IQR = Q3 - Q1
    outlier_mask = (diff < Q1 - 1.5 * IQR) | (diff > Q3 + 1.5 * IQR)

    print(f"Number of outliers for {construct}: {outlier_mask.sum()}")
    
    df_outliers_removed.loc[outlier_mask, [f"{construct}_virtual_human", f"{construct}_chatbot"]] = np.nan

    sns.boxplot(diff, ax=axes[i])
    axes[i].set_title(construct_abb[construct])

fig.supylabel("Score VH − Score Chatbot")
fig.suptitle("Boxplot of Differences")
plt.savefig("figures/boxplot_differences.pdf", bbox_inches="tight", transparent=True, pad_inches=0.5)

### Normality Check

In [ ]:
# Check normality by plotting the distributions and the Shapiro-Wilk test
normality_results = []
fig, axes = plt.subplots(2, 2, figsize=(WIDTH, 8))
axes = axes.flatten()

for i, construct in enumerate(hypothesis_constructs):
    diff = df_constructs[f"{construct}_virtual_human"] - df_constructs[f"{construct}_chatbot"] # Difference of construct score between conditions
    stat, p = shapiro(diff)
    
    normality_results.append({
        "Construct": construct,
        "W": stat,
        "p": p,
        "normal": p > 0.05
    })

    sns.histplot(diff, kde=True, ax=axes[i])
    axes[i].set_title(f"{construct_abb[construct]}")

fig.suptitle("Histogram of Differences (VH − Chatbot)")
plt.savefig("figures/histograms.pdf", bbox_inches="tight", transparent=True, pad_inches=0.5)
normality_df = pd.DataFrame(normality_results)
normality_df

## Paired t-testing (Test Statistics)

### Results With Outliers

In [ ]:
results = []

for construct in hypothesis_constructs:
    # get paired data of for each construct 
    paired_data = df_constructs[[f"{construct}_virtual_human", f"{construct}_chatbot"]]

    # get number of valid participant data
    n_participants = len(paired_data)

    # means and stds
    chatbot_mean = paired_data[f"{construct}_chatbot"].mean()
    chatbot_sd = paired_data[f"{construct}_chatbot"].std()

    vh_mean = paired_data[f"{construct}_virtual_human"].mean()
    vh_sd = paired_data[f"{construct}_virtual_human"].std()

    # paired t-testing (VH − chatbot)
    t_stat, p_value = stats.ttest_rel(
        paired_data[f"{construct}_virtual_human"],
        paired_data[f"{construct}_chatbot"]
    )

    # Cohen's d (VH − chatbot)
    diff = paired_data[f"{construct}_virtual_human"] - paired_data[f"{construct}_chatbot"]
    cohens_d = diff.mean() / diff.std(ddof=1)

    results.append({
        "Construct": construct,
        "Participants": n_participants,
        "VH Mean": vh_mean,
        "VH SD": vh_sd,
        "Chatbot Mean": chatbot_mean,
        "Chatbot SD": chatbot_sd,
        "t": t_stat,
        "p": p_value,
        "sig": p_value < 0.05,
        "Cohen's d": cohens_d
    })

results_df = pd.DataFrame(results)
results_df

### Results Without Outliers

In [ ]:
results = []

for construct in hypothesis_constructs:
    # get paired data of for each main construct 
    paired_data = df_outliers_removed[[f"{construct}_virtual_human", f"{construct}_chatbot"]].dropna()
    
    # get number of valid participant data
    n_participants = len(paired_data)
    
    # means and stds
    vh_mean = paired_data[f"{construct}_virtual_human"].mean()
    vh_sd = paired_data[f"{construct}_virtual_human"].std()
    chatbot_mean = paired_data[f"{construct}_chatbot"].mean()
    chatbot_sd = paired_data[f"{construct}_chatbot"].std()

    # paired t-testing (VH − chatbot)
    t_stat, p_value = stats.ttest_rel(
        paired_data[f"{construct}_virtual_human"],
        paired_data[f"{construct}_chatbot"]
    )

    # Cohen's d (VH − chatbot)
    diff = paired_data[f"{construct}_virtual_human"] - paired_data[f"{construct}_chatbot"]
    cohens_d = diff.mean() / diff.std(ddof=1)

    results.append({
        "Construct": construct,
        "Participants": n_participants,
        "VH Mean": vh_mean, "VH SD": vh_sd,
        "Chatbot Mean": chatbot_mean, "Chatbot SD": chatbot_sd,
        "t": t_stat, "p": p_value,
        "sig": p_value < 0.05,
        "Cohen's d": cohens_d
    })

results_df = pd.DataFrame(results)
results_df

## Exploratory Variables

In [ ]:
# define exploratory constructs and create dataframe
exploratory_constructs = ["MC", "UAA"]
exploratory_cols = [f"{construct}_{condition}" for construct in exploratory_constructs for condition in ["virtual_human", "chatbot"]]
df_exploratory_constructs = df_constructs[exploratory_cols]
df_exploratory_constructs.describe()

In [ ]:
# Create boxplots of exploratory constructs by condition
fig, axes = plt.subplots(1, 2, figsize=(WIDTH, 6))
axes = axes.flatten()

for i, construct in enumerate(exploratory_constructs):
    paired_data = df_exploratory_constructs[[f"{construct}_virtual_human", f"{construct}_chatbot"]].rename(columns={
        f"{construct}_virtual_human": "Virtual Human",
        f"{construct}_chatbot": "Chatbot"
    })

    sns.boxplot(data=paired_data, ax=axes[i])
    
    axes[i].set_title(f"{construct_abb[construct]}")
    axes[i].set_ylim(-3.5, 3.5)

fig.supylabel("Value")
fig.supxlabel("Conversational Agent")
plt.tight_layout()
plt.savefig("figures/boxplots_exploratory.pdf", bbox_inches="tight", transparent=True, pad_inches=0.5)

plt.show()

## Paired T-test Assumption Check

### Outlier Detection

In [ ]:
# Visualize outliers and detect outliers by IQR method
fig, axes = plt.subplots(1, 2, figsize=(WIDTH, 5))
axes = axes.flatten()
df_outliers_removed = df_exploratory_constructs.copy()

for i, construct in enumerate(exploratory_constructs):
    diff = df_exploratory_constructs[f"{construct}_virtual_human"] - df_exploratory_constructs[f"{construct}_chatbot"]
    
    Q1, Q3 = diff.quantile(0.25), diff.quantile(0.75)
    IQR = Q3 - Q1
    outlier_mask = (diff < Q1 - 1.5 * IQR) | (diff > Q3 + 1.5 * IQR)

    print(f"Number of outliers for {construct} detected: {outlier_mask.sum()}")
    
    df_outliers_removed.loc[outlier_mask, [f"{construct}_virtual_human", f"{construct}_chatbot"]] = np.nan

    sns.boxplot(diff, ax=axes[i])
    axes[i].set_title(construct_abb[construct])

fig.supylabel("Score VH − Score Chatbot")
fig.suptitle("Boxplot of Differences")
plt.savefig("figures/exploratory_boxplot_differences.pdf", bbox_inches="tight", transparent=True, pad_inches=0.5)

### Normality Check

In [ ]:
# Check normality by plotting the distributions and the Shapiro-Wilk test
normality_results = []
fig, axes = plt.subplots(1, 2, figsize=(WIDTH, 5))
axes = axes.flatten()

for i, construct in enumerate(exploratory_constructs):
    diff = df_exploratory_constructs[f"{construct}_virtual_human"] - df_exploratory_constructs[f"{construct}_chatbot"]
    stat, p = shapiro(diff)
    
    normality_results.append({
        "Construct": construct,
        "W": stat,
        "p": p,
        "normal": p > 0.05
    })

    sns.histplot(diff, kde=True, ax=axes[i])
    axes[i].set_title(f"{construct_abb[construct]}")

fig.suptitle("Histogram of Differences (VH − Chatbot)")
plt.savefig("figures/exploratory_histograms.pdf", bbox_inches="tight", transparent=True, pad_inches=0.5)
normality_df = pd.DataFrame(normality_results)
normality_df

## Paired t-testing (Test Statistics)

### Results with Outliers

In [ ]:
results = []

for construct in exploratory_constructs:
    # get paired data of for each exploratory construct
    paired_data = df_exploratory_constructs[[f"{construct}_virtual_human", f"{construct}_chatbot"]]

    # get number of valid participant data
    n_participants = len(paired_data)

    # means and stds
    chatbot_mean = paired_data[f"{construct}_chatbot"].mean()
    chatbot_sd = paired_data[f"{construct}_chatbot"].std()

    vh_mean = paired_data[f"{construct}_virtual_human"].mean()
    vh_sd = paired_data[f"{construct}_virtual_human"].std()

    # paired t-testing (VH − chatbot)
    t_stat, p_value = stats.ttest_rel(
        paired_data[f"{construct}_virtual_human"],
        paired_data[f"{construct}_chatbot"]
    )

    # Cohen's d (VH − chatbot)
    diff = paired_data[f"{construct}_virtual_human"] - paired_data[f"{construct}_chatbot"]
    cohens_d = diff.mean() / diff.std(ddof=1)

    results.append({
        "Construct": construct,
        "Participants": n_participants,
        "VH Mean": vh_mean,
        "VH SD": vh_sd,
        "Chatbot Mean": chatbot_mean,
        "Chatbot SD": chatbot_sd,
        "t": t_stat,
        "p": p_value,
        "sig": p_value < 0.05,
        "Cohen's d": cohens_d
    })

results_df = pd.DataFrame(results)
results_df

### Results without Outliers

In [ ]:
results = []

for construct in exploratory_constructs:
    # get paired data of for each construct 
    paired_data = df_outliers_removed[[f"{construct}_virtual_human", f"{construct}_chatbot"]].dropna()
    
    # get number of valid participant data
    n_participants = len(paired_data)

    # means and stds
    vh_mean = paired_data[f"{construct}_virtual_human"].mean()
    vh_sd = paired_data[f"{construct}_virtual_human"].std()
    chatbot_mean = paired_data[f"{construct}_chatbot"].mean()
    chatbot_sd = paired_data[f"{construct}_chatbot"].std()

    # paired t-testing (VH − chatbot)
    t_stat, p_value = stats.ttest_rel(
        paired_data[f"{construct}_virtual_human"],
        paired_data[f"{construct}_chatbot"]
    )

    # Cohen's d (VH − chatbot)
    diff = paired_data[f"{construct}_virtual_human"] - paired_data[f"{construct}_chatbot"]
    cohens_d = diff.mean() / diff.std(ddof=1)

    results.append({
        "Construct": construct,
        "Participants": n_participants,
        "VH Mean": vh_mean, "VH SD": vh_sd,
        "Chatbot Mean": chatbot_mean, "Chatbot SD": chatbot_sd,
        "t": t_stat, "p": p_value,
        "sig": p_value < 0.05,
        "Cohen's d": cohens_d
    })

results_df = pd.DataFrame(results)
results_df